# # Pronóstico de la volatilidad diaria del petróleo WTI: EGARCH(1,1,1)-t con media constante vs. Random Forest con rezagos

**Proyecto final — Aprendizaje Supervisado · Maestría en Inteligencia Artificial (USFQ)**

- **Integrante:** Darlyn Ludeña
- **Repositorio:** https://github.com/darlamai/proyecto_final_ml_wti.git
- **Datos:** Yahoo Finance, ticker `CL=F` (serie continua de futuros WTI), precios diarios del 04-ene-2010 al 30-oct-2026: precio de cierre (`WTI`), máximo (`WTI_High`) y mínimo (`WTI_Low`) de cada sesión. Se eliminan las sesiones del 20 y 21 de abril de 2020 por presentar precios negativos (vencimiento del contrato de mayo de 2020).
  - **Retornos:** se calculan a partir del precio de cierre como retornos logarítmicos diarios en porcentaje, $r_t = 100 \cdot \ln(P_t / P_{t-1})$. Son la serie sobre la que se realiza el diagnóstico (EDA), se estima el modelo EGARCH(1,1,1)-t con media constante y se construyen las variables explicativas del Random Forest.
  - **Volatilidad de Parkinson:** se calcula a partir de los precios máximo y mínimo de cada sesión mediante el estimador de rango de Parkinson, $\sigma^2_{P,t} = \left[100 \cdot \ln(\text{High}_t / \text{Low}_t)\right]^2 / (4 \ln 2)$, expresado en $\%^2$. Es el proxy de la varianza diaria y la variable objetivo con la que se evalúan los pronósticos de ambos modelos.[5]

Esta notebook es el documento principal del proyecto: contiene la introducción, la descripción metodológica y ejecuta, en orden, las notebooks de cada etapa.

| Ítem de la rúbrica | Pts | Dónde se desarrolla |
|---|---|---|
| Introducción | 0.1 | Sección 1 (este notebook)  [Introducción](#introduccion) |
| EDA | 0.6 | `01_eda_raw` y `02_eda_returns` [EDA raw](notebooks/01.eda_raw.ipynb) y [EDA returns](notebooks/02.eda_returns.ipynb)|
| Feature Extraction | 0.25 | `04_feature_extraction`  [Feature extraction](notebooks/04.feature_extraction.ipynb) |
| Feature Selection | 0.25 |  `05_feature_selection` [Feature selection](notebooks/05.feature_selection.ipynb)  |
| Pipelines | 0.5 | `06_random_forest` [Random Forest](notebooks/06.random_forest.ipynb)  |
| Optimización de hiperparámetros | 3.5 | `06_random_forest` [Random Forest: Optimización con Optuna](notebooks/06.random_forest.ipynb#optuna) |
| Comparación estadística de 2 técnicas | 3.5 | `07_comparison_model` [Comparison model](notebooks/07.comparison_model.ipynb). Modelo econométrico: `03_modelo_base`  [EGARCH con media constante](notebooks/03.modelo_base.ipynb) |
| Visualización con t-SNE | 1 | `05_feature_selection` [Feature selection](notebooks/05.feature_selection.ipynb)  |
| Idioma | 0.1 | Todo el proyecto  |
| Conclusiones | 0.1 | En este notebook y en los notebooks que lo requieran |
| Referencias | 0.1 | Comentarios con el prompt de chatgpt en los scripts de python `\src` y referencias en formato IEEE de los conceptos teóricos cuando lo requiera y usando citas en los `\notebooks` |


<a id="introduccion"></a>


## 1. Introducción

### 1.1 Motivación


El petróleo es uno de los activos más relevantes de la economía mundial y uno de los más volátiles. Sus precios reaccionan a choques de oferta (decisiones de la OPEP+, conflictos geopolíticos), de demanda (recesiones, la pandemia de COVID-19) y a restricciones físicas del mercado, como la capacidad de almacenamiento que llevó al contrato WTI de mayo de 2020 a cerrar en precio negativo.

Para quienes toman decisiones sobre este mercado, anticipar **cuánto** puede moverse el precio suele ser tan importante como anticipar **hacia dónde** se moverá. La volatilidad es un insumo directo para:

- la **gestión de riesgos** (Valor en Riesgo, límites de exposición, requerimientos de margen),
- la valoración de **coberturas y derivados** sobre el crudo,
- la **planificación fiscal y presupuestaria** de economías dependientes del petróleo como la ecuatoriana, cuyos ingresos públicos y balanza comercial son sensibles al precio del crudo, que se cotiza con referencia al WTI.


### 1.2 Problema y pregunta de investigación
Los retornos diarios del petróleo son prácticamente impredecibles a partir de su propio pasado, pero su **volatilidad sí presenta dependencia temporal**: los periodos de alta incertidumbre tienden a agruparse (*volatility clustering*). Desde la econometría, este fenómeno se modela con la familia ARCH/GARCH/EGARCH [1], [2], [8], que sigue siendo una referencia difícil de superar [3]. Desde el aprendizaje automático, modelos no lineales como Random Forest [4] pueden aprender relaciones entre la volatilidad pasada y la futura sin imponer una forma funcional.

> **Pregunta de investigación:** ¿un modelo de machine learning (Random Forest) basado en rezagos de medidas de volatilidad pronostica la volatilidad diaria del WTI con mayor precisión que un modelo econométrico EGARCH(1,1,1) con media constante y errores t de Student, y es esa diferencia estadísticamente significativa?

- $H_0$: ambos modelos tienen la misma precisión de pronóstico de la volatilidad del día siguiente.
- $H_1$: la precisión de ambos modelos es distinta.



### 1.3 Enfoque

1. **Preparación y EDA** de la serie: calidad de datos, transformación y diagnóstico de estacionariedad, estacionalidad y dependencia temporal. El diagnóstico (efectos ARCH significativos y colas pesadas) justifica el modelo base.
2. **Modelo base (econométrico):** se comparan tres especificaciones EGARCH(1,1,1) con errores t (AR(1) sin constante, AR(1) con constante y media constante) y se elige la de media constante, que entrega un pronóstico de la varianza condicional a un paso. [8]
3. **Modelo de machine learning:** Random Forest con 67 variables candidatas construidas por rezagos (rezagos de $r_t^2$, $|r_t|$ y del estimador de Parkinson, volatilidades y promedios móviles), selección tipo *filter* por información mutua y optimización de hiperparámetros con Optuna. [4], [10]
4. **Validación:** validación cruzada para series de tiempo (*walk-forward* con ventana expansiva), 2 repeticiones × 5 folds, dentro del periodo de desarrollo 2010–2024. Se reserva enero 2025 – octubre 2026 como test final.
5. **Comparación:** los dos modelos se evalúan en los mismos folds con las funciones de pérdida QLIKE (principal) y MSE [6], y se comparan con la prueba de Wilcoxon [9] y la prueba de Diebold-Mariano [7]. Finalmente, ambos se evalúan una sola vez en el test.



### 1.4 Estructura del proyecto


El proyecto se organiza en siete notebooks que se ejecutan en orden. Cada uno guarda sus resultados en `data/processed/` o `results/`, y el siguiente los carga.

| Notebook | Etapa | Pregunta que responde | Resultado principal |
|---|---|---|---|
| `01.eda_raw` | Preparación y división | ¿Los datos son confiables y cómo se dividen sin fuga de información? | Serie limpia, retornos y Parkinson; desarrollo 2010–2024, test 2025–2026, 10 folds |
| `02.eda_returns` | Diagnóstico | ¿Qué estructura tiene la serie y qué modelo base conviene? | Retornos estacionarios, sin estacionalidad, con fuertes efectos ARCH |
| `03.modelo_base` | Modelo econométrico | ¿Qué especificación GARCH es adecuada y cómo pronostica? | Media constante–EGARCH(1,1,1)-t; QLIKE de validación 2,866 |
| `04.feature_extraction` | Variables predictoras | ¿Qué información del pasado puede usar el modelo de ML? | 67 variables construidas por rezagos |
| `05.feature_selection` | Selección y t-SNE | ¿Cuántas y cuáles variables conviene usar? | k = 16 variables, todas medidas de volatilidad |
| `06.random_forest` | Modelo de ML | ¿Qué hiperparámetros minimizan el error de validación? | RF optimizado con Optuna |
| `07.comparison_model` | Comparación | ¿Un modelo pronostica significativamente mejor que el otro? | No hay diferencia significativa (Wilcoxon *p* > 0,05; DM *p* > 0,05) |

## 2. Ejecución de las etapas del proyecto
Cada notebook se ejecuta en orden y comparte sus resultados a través de `data/processed/`.

In [1]:
from pathlib import Path
RAIZ = Path.cwd()
assert (RAIZ / "notebooks").exists(), "Ejecutar esta notebook desde la raíz del proyecto"

In [2]:
RAIZ

WindowsPath('c:/Users/ASUS/Desktop/Maestria IA USFQ/AprendizajeSupervisado/Proyectofinal_ml_wti')

### 2.1 Carga de la data cruda y EDA

 `01.eda_raw`: calidad de datos, transformación y división temporal

**Objetivo.** Verificar la calidad de la serie descargada, transformarla en las variables del proyecto y definir la división temporal antes de tomar cualquier decisión de modelado.

**Metodología.**
- **Datos:** 4.233 precios diarios del futuro WTI (`CL=F`, Yahoo Finance), del 4-ene-2010 al 30-oct-2026, con precio de cierre, máximo y mínimo de cada sesión.
- **Calidad:** revisión de faltantes, valores no positivos, días sin variación, huecos en el calendario (frente a los feriados de EE. UU.) y saltos extremos. Los precios se contrastaron con la serie de futuros publicada por la EIA.
- **Transformación:** retornos logarítmicos en porcentaje, $r_t = 100\ln(P_t/P_{t-1})$, y estimador de volatilidad de Parkinson, $\sigma^2_{P,t} = [100\ln(H_t/L_t)]^2/(4\ln 2)$, como proxy de la varianza diaria.
- **División temporal:** no se usa k-fold aleatorio porque introduciría información futura en el entrenamiento. Se aplica *Time Series Cross-Validation* con ventana expansiva.

**Resultados.**
- No hay datos faltantes ni infinitos. Se identificaron 14 días sin variación en el precio, consistentes con los datos de la EIA, por lo que se conservaron.
- Solo tres días hábiles no tienen dato, y corresponden a cierres extraordinarios del mercado: el huracán Sandy (29 y 30 de octubre de 2012) y el duelo nacional por George H. W. Bush (5 de diciembre de 2018). No se imputan.
- Los saltos más extremos se concentran entre marzo y mayo de 2020, durante el colapso de la demanda por la pandemia.
- El 20 de abril de 2020 el contrato de mayo cerró en −37,63 USD, y el 21 de abril tuvo un mínimo intradía de −16,74 USD. Como el logaritmo no está definido para precios ≤ 0, se eliminan ambas sesiones (2 de 4.233).
- Se eligieron los retornos logarítmicos porque no dependen de la escala del precio, se interpretan como variación porcentual aproximada y son la variable estándar en los modelos de la familia GARCH.
- Además de 2020, la tabla de saltos extremos muestra movimientos grandes del precio máximo en marzo de 2026, dentro del periodo de test.



- **División:**

| Conjunto | Periodo | Observaciones | Uso |
|---|---|---|---|
| Desarrollo | ene 2010 – dic 2024 | 3.769 | Entrenamiento y validación en 10 folds (2 repeticiones × 5, validación de 12 meses) |
| Tramo de diagnóstico | ene 2010 – jun 2019 | 2.385 | Decisiones del modelo base; es el entrenamiento más corto, común a todos los folds |
| Test final | ene 2025 – oct 2026 | 461 | Evaluación final, usada una sola vez |

Los bloques de validación cubren regímenes distintos: COVID-19 (2020), recuperación (2021), guerra en Ucrania (2022) y años más estables (2023–2024).

In [3]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/01.eda_raw.ipynb"

[NbConvertApp] Converting notebook ./notebooks/01.eda_raw.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 444190 bytes to notebooks\01.eda_raw.ipynb


### 2.2 EDA de la data transformada en retornos

`02.eda_returns`: diagnóstico de los retornos y elección del modelo base

**Objetivo.** Identificar la estructura estadística de los retornos para elegir el modelo econométrico. Todas las pruebas usan solo el tramo de diagnóstico (ene 2010 – jun 2019, 2.385 observaciones), para que ninguna decisión use datos de validación o de test.

**Metodología.** Estadística descriptiva y prueba de Jarque-Bera; estacionariedad con ADF, PP y KPSS; estacionalidad mensual con Kruskal-Wallis; dependencia en la media con ACF, PACF, Ljung-Box y criterios AIC/BIC; dependencia en la varianza con la ACF de $r_t^2$, Ljung-Box y la prueba ARCH-LM.

**Resultados.**
- **Distribución:** retorno medio diario de −0,014 % y volatilidad diaria de 2,05 %. La distribución es casi simétrica (asimetría −0,017) pero con colas pesadas (curtosis en exceso 2,92); Jarque-Bera rechaza la normalidad. → Se justifica una distribución **t de Student**.
- **Estacionariedad:** ADF (−34,40) y PP (−51,96) rechazan la raíz unitaria y KPSS (0,079; *p* = 0,10) no rechaza la estacionariedad. → Se modelan los **retornos**.
- **Estacionalidad:** Kruskal-Wallis no encuentra diferencias entre meses (*p* = 0,504). → No se incluyen términos estacionales.
- **Media:** dependencia débil. Ljung-Box es significativo hasta el rezago 5 (*p* = 0,002), pero la PACF solo muestra un coeficiente relevante en el rezago 1 (−0,065). BIC elige AR(1) y AIC elige AR(2).
- **Volatilidad:** dependencia muy fuerte. Ljung-Box sobre $r_t^2$ tiene *p* ≈ 0 en todos los rezagos y la prueba ARCH-LM rechaza la ausencia de efectos ARCH (estadístico 296,4; *p* ≈ 0). La volatilidad móvil muestra agrupamiento, con picos en 2015–2016.
- **Proxy de Parkinson:** media de 3,88 %² y mediana de 2,25 %², con una cola derecha marcada (máximo 54,99 %²).

**Conclusión.** Los retornos son estacionarios, sin estacionalidad, con una dependencia débil en la media, una fuerte dependencia en la varianza y colas pesadas. Esto lleva a proponer un modelo de la familia **GARCH con errores t**, que se especifica en el notebook 03.

In [4]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/02.eda_returns.ipynb"

[NbConvertApp] Converting notebook ./notebooks/02.eda_returns.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 316786 bytes to notebooks\02.eda_returns.ipynb


### 2.3 Modelo econométrico EGARCH con media constante

`03.modelo_base`: modelo econométrico EGARCH con media constante

**Objetivo.** Especificar, verificar y evaluar el modelo base, que pronostica la varianza condicional del día siguiente.

**Metodología.**
- Se compararon tres especificaciones EGARCH(1,1,1) con errores t, estimadas en el tramo de diagnóstico (2.384 observaciones): AR(1) sin constante, AR(1) con constante y media constante. Se eligió EGARCH porque modela el logaritmo de la varianza (positiva por construcción) e incluye un término de asimetría.
- La selección consideró los criterios AIC/BIC y la verificación de supuestos sobre los residuos estandarizados $z_t$: Ljung-Box sobre $z_t$ y $z_t^2$, ARCH-LM, ajuste a la distribución t (Kolmogorov-Smirnov y QQ-plot) y *sign bias test* de Engle y Ng.
- Evaluación en los 10 folds: en cada uno, el modelo se estima con su entrenamiento y pronostica a un paso cada día de la validación, con parámetros fijos. Los pronósticos se comparan con Parkinson mediante QLIKE (métrica principal) y MSE.

**Resultados.**
- **Modelo elegido: media constante–EGARCH(1,1,1)-t.** Su AIC/BIC (9.561,3 / 9.595,9) es apenas 1,6 puntos mayor que el del AR(1) sin constante, pero tiene los mejores diagnósticos residuales y evita un término AR que no fue significativo.
- **Supuestos:** sin autocorrelación en $z_t$ ni en $z_t^2$ (Ljung-Box, *p* > 0,05), sin efectos ARCH remanentes (ARCH-LM, *p* = 0,106), ajuste aceptable a la t (KS, *p* = 0,302; ν ≈ 8,6) y sin sesgo de signo significativo al 5 % (*p* = 0,097).
- **Parámetros:** volatilidad muy persistente (β ≈ 0,99) y γ < 0 en todos los folds: las caídas del precio elevan más la volatilidad que las subidas.
- **Validación (10 folds):** QLIKE media de **2,866** (desv. 0,547) y MSE medio de 989,2. El error es mayor en los folds con la crisis de 2020 (QLIKE 3,59 y 3,41) y con 2022 (3,45), y menor en 2024 (2,11). Los parámetros son estables al expandir el entrenamiento (β entre 0,984 y 0,992).

In [5]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/03.modelo_base.ipynb"

[NbConvertApp] Converting notebook ./notebooks/03.modelo_base.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 1427612 bytes to notebooks\03.modelo_base.ipynb


### 2.4 Feature Extraction

`04.feature_extraction`: construcción de variables por rezagos

**Objetivo.** Construir las variables predictoras del Random Forest. Para cada día objetivo τ, las variables usan solo información hasta τ−1 y la etiqueta es el estimador de Parkinson del día τ.

**Metodología.** Extracción clásica por rezagos, el enfoque adecuado para una serie temporal estructurada (no se usa *transfer learning*):

| Grupo | Variables | Cantidad |
|---|---|---|
| Rezagos de la varianza | $r^2_{\tau-1}, \dots, r^2_{\tau-20}$ | 20 |
| Rezagos de la magnitud | $\lvert r_{\tau-1}\rvert, \dots, \lvert r_{\tau-20}\rvert$ | 20 |
| Rezagos de Parkinson | $\sigma^2_{P,\tau-1}, \dots, \sigma^2_{P,\tau-20}$ | 20 |
| Volatilidad móvil | Desviación estándar de $r$ en 5, 21 y 63 días | 3 |
| Parkinson promedio | Media de $\sigma^2_P$ en 5, 21 y 63 días | 3 |
| Retorno con signo | $r_{\tau-1}$ (efecto apalancamiento) | 1 |

Como los rezagos y promedios móviles son fórmulas fijas que solo usan el pasado, se construyen sin fuga de información. La alineación se verificó para fechas específicas.

**Resultados.**
- **67 variables candidatas.** Desarrollo: 3.706 observaciones (7-abr-2010 a 31-dic-2024; se descartan los primeros 63 días por falta de historial). Test: 461 observaciones, con sus rezagos calculados a partir de diciembre de 2024.
- - Las variables más asociadas con el Parkinson del día siguiente (correlación de Spearman en el tramo de diagnóstico, 2010 – jun 2019) son los promedios de Parkinson de 21 y 5 días  y la volatilidad de 21 días. Hay alta redundancia entre variables, en especial entre los rezagos de Parkinson y las medidas móviles.

In [6]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/04.feature_extraction.ipynb"

[NbConvertApp] Converting notebook ./notebooks/04.feature_extraction.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 327992 bytes to notebooks\04.feature_extraction.ipynb


### 2.5 Feature Selection

`05.feature_selection`: selección de variables y visualización con t-SNE

**Objetivo.** Elegir cuántas y cuáles variables usa el Random Forest, y visualizar el espacio de características.

**Metodología.**
- **Selección tipo filter:** información mutua (`mutual_info_regression`), que capta relaciones lineales y no lineales. Se aplica con `SelectKBest` como primer paso del pipeline, ajustado **solo con el entrenamiento de cada fold**.
- **Elección de k:** se evaluaron valores entre 5 y 67 en los 10 folds, con un Random Forest de referencia, comparando QLIKE y MSE de validación.
- **t-SNE:** proyección a dos dimensiones de las variables seleccionadas (escaladas), coloreando cada día por su nivel de Parkinson.

**Resultados.**
- **k = 16.** Obtiene el menor MSE promedio (1.349,1) y una QLIKE (2,8713) prácticamente igual al mínimo (2,8692 con k = 45). Con menos variables se obtiene el mismo desempeño, porque muchas son redundantes.
- **Variables elegidas:** todas son medidas de volatilidad. Rezagos de Parkinson (días 1 a 9 y 18), volatilidad móvil de 5, 21 y 63 días, y Parkinson promedio de 5, 21 y 63 días. No se eligió ningún rezago de $r^2$ ni de $\lvert r\rvert$, ni el retorno con signo: el estimador de Parkinson aporta más información que el retorno al cuadrado.
- Las de mayor información mutua son el Parkinson promedio de 5 días (0,33) y de 21 días (0,32).
- **t-SNE:** se observan días de volatilidad baja en una zona del mapa y de volatilidad alta en otra, más una zona intermedia mezclada. Las variables contienen información sobre la volatilidad del día siguiente, aunque no la separan por completo.

In [7]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/05.feature_selection.ipynb"

[NbConvertApp] Converting notebook ./notebooks/05.feature_selection.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 504534 bytes to notebooks\05.feature_selection.ipynb


### 2.6 Random Forest

`06.random_forest`: Random Forest con rezagos

**Objetivo.** Optimizar y evaluar el modelo de machine learning en los mismos 10 folds que el EGARCH.

#### Pipeline de scikit-learn

El modelo se implementa como un `Pipeline` de scikit-learn que encadena la selección de variables y el estimador en un solo objeto. Se construye con la función `crear_pipeline` del módulo `src/random_forest.py`, de modo que el mismo pipeline se reutiliza en la optimización, las curvas de diagnóstico, la evaluación por fold y el test final.

```python
Pipeline([
    ("seleccion", SelectKBest(score_func=partial(mutual_info_regression, random_state=42), k=16)),
    ("modelo",    RandomForestRegressor(n_estimators=350, max_depth=7, min_samples_leaf=10,
                                        max_features=0.96, random_state=42, n_jobs=-1)),
])
```

| Paso | Clase de scikit-learn | Qué hace | Qué aprende en `.fit` |
|---|---|---|---|
| `seleccion` | `SelectKBest` con `mutual_info_regression` | Calcula la información mutua entre cada una de las 67 variables candidatas y el Parkinson del día τ, y conserva las 16 con mayor puntaje (selección tipo *filter*) | Los puntajes de información mutua y cuáles son las 16 variables elegidas |
| `modelo` | `RandomForestRegressor` | Promedia árboles de regresión entrenados sobre muestras *bootstrap*; cada división evalúa una fracción aleatoria (`max_features`) de las variables seleccionadas | La estructura de los árboles (umbrales y valores de las hojas) |

**Cómo se usa en cada fold:**
1. `pipeline.fit(X_train, y_train)`: ambos pasos se ajustan **solo con el entrenamiento del fold**. La información mutua y las 16 variables se eligen con esos días, y el Random Forest se entrena con ellas.
2. `pipeline.predict(X_val)`: se aplican a la validación la **misma** selección y el **mismo** modelo ya ajustados, sin volver a calcular nada. Así, ninguna información de la validación influye en el entrenamiento.
3. 3. Los pronósticos se limitan a un mínimo positivo (`H_MIN`), necesario para calcular la QLIKE.

**Decisiones de diseño:**
- **Sin escalado:** el Random Forest divide por umbrales, así que la escala de las variables no cambia el resultado. Un `StandardScaler` no aportaría nada.
- **Etiqueta sin transformar:** el pipeline se entrena directamente sobre Parkinson (%²), sin logaritmo. El MSE y la QLIKE se minimizan cuando el pronóstico es la media condicional de la varianza; con logaritmo, el modelo la subestimaría de forma sistemática. La influencia de los días extremos se controla con `min_samples_leaf`.
- **Reproducibilidad:** la información mutua y el Random Forest usan `random_state = 42`.
- **Eficiencia en la búsqueda:** como la información mutua de cada fold no depende de los hiperparámetros del Random Forest, durante Optuna se calcula una vez por fold y se pasa a `SelectKBest` mediante una función de puntajes fijos. El resultado es idéntico a recalcularla en cada prueba, pero evita repetir el mismo cálculo cientos de veces. En la evaluación final por fold y en el test se usa el pipeline completo, con la información mutua calculada dentro del `.fit`.
- **Mismo pipeline en todo el proyecto:** la optimización, las curvas de entrenamiento y validación, la curva de aprendizaje, los errores por fold y el test final (`.fit` con todo el desarrollo 2010–2024 y `.predict` sobre 2025–2026) usan el mismo objeto con los mismos pasos.

#### Optimización y diagnóstico

- **Optimización con Optuna:** 50 pruebas con búsqueda bayesiana (TPE) y poda por mediana, minimizando la QLIKE promedio de los 10 folds de *Time Series Cross-Validation*. Se ajustaron los hiperparámetros del paso `modelo`: `n_estimators` (100–600), `max_depth` (2–20), `min_samples_leaf` (5–200, escala logarítmica) y `max_features` (0,2–1,0). El número de variables del paso `seleccion` se fijó en k = 16, elegido con la misma validación en el notebook 05.
- **Diagnóstico:** curvas de entrenamiento y validación para cada hiperparámetro y curva de aprendizaje.

**Resultados.**
- **Mejores hiperparámetros:** `n_estimators = 350`, `max_depth = 7`, `min_samples_leaf = 10`, `max_features ≈ 0,96`, con una QLIKE de validación de **2,849**. De las 50 pruebas, 34 se completaron y 16 se podaron.
- **Importancia de los hiperparámetros:** `min_samples_leaf` explica la mayor parte de las diferencias entre pruebas (0,73), seguido de `n_estimators` (0,19).
- **Curvas:** la QLIKE de validación mejora hasta una profundidad de 6–7 y luego se estabiliza; hojas más grandes producen subajuste (el error sube en entrenamiento y validación); el número de árboles no cambia el desempeño. Los hiperparámetros elegidos están en la zona de menor error de validación.
- **Curva de aprendizaje:** la QLIKE de validación baja de 3,20 a 2,89 al pasar de 500 a 1.000 días de entrenamiento y luego se estabiliza en ≈ 2,86.
- **Errores por fold:** QLIKE media de 2,21 en entrenamiento y 2,85 en validación. Los errores son mayores en 2020 (3,96 y 3,80), donde el modelo subestima los picos de volatilidad.
- **Importancia de variables:** el Parkinson del día anterior (0,49) y el promedio de Parkinson de 5 días (0,25) concentran la mayor parte de la importancia.

In [8]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/06.random_forest.ipynb"

[NbConvertApp] Converting notebook ./notebooks/06.random_forest.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 914961 bytes to notebooks\06.random_forest.ipynb


### 2.7 Comparación estadística de modelos

`07.comparison_model`: comparación estadística y test final

**Objetivo.** Determinar si un modelo pronostica la volatilidad significativamente mejor que el otro.

**Metodología.**
- Se verificó que ambos modelos se evalúan en los mismos 10 folds, en los mismos 2.512 días-fold y contra el mismo Parkinson observado.
- **Prueba de Wilcoxon** de rangos con signo (y *t* pareada) sobre las pérdidas de los 10 folds, como pide la rúbrica.
- **Prueba de Diebold-Mariano** sobre las pérdidas diarias, con varianza de Newey-West, por repetición.
- **Referencia ingenua:** el promedio de Parkinson de los últimos 21 días.
- **Test final:** ambos modelos se estiman con todo el desarrollo (2010–2024) y se evalúan una sola vez en 2025–2026.

**Resultados.**

| | EGARCH media const. | Random Forest | Ingenuo (21 días) |
|---|---|---|---|
| QLIKE entrenamiento | 2,374 | 2,297 | 2,379 |
| QLIKE validación (10 folds) | 2,866 | 2,849 | 2,829 |
| QLIKE test | 3,239 | **2,839** | 2,974 |
| Folds con menor QLIKE (vs. el otro modelo) | 3 | 7 | — |

- **Validación:** el Random Forest fue mejor en 7 de los 10 folds, sobre todo entre 2021 y 2024; el EGARCH fue mejor en los dos folds que incluyen la crisis de 2020. Con el MSE, el EGARCH tiene menor error medio (989 frente a 1.324).
- **Pruebas:** no hay diferencias significativas al 5 %. Wilcoxon: QLIKE *p* = 0,492 y MSE *p* = 0,084. Diebold-Mariano: QLIKE *p* = 0,724 y 0,853; MSE *p* = 0,092 y 0,089.
- **Referencia ingenua:** ninguno de los modelos la supera de forma significativa (Diebold-Mariano, *p* > 0,05).
- **Test final:** el Random Forest obtuvo las menores pérdidas (QLIKE 2,839; MSE 729,4), sin diferencias significativas (*p* ≥ 0,146). Su error de test es casi igual al de validación, mientras que el del EGARCH empeora (de 2,866 a 3,239).

**Conclusión.** No se rechaza $H_0$: no hay evidencia suficiente para afirmar que un modelo tenga mayor precisión que el otro, lo que no demuestra que sean equivalentes. El Random Forest muestra una ventaja descriptiva pequeña y un error de test casi igual al de validación. En validación fue mejor en los periodos de volatilidad moderada y el EGARCH en los folds de 2020; en el test, en cambio, el Random Forest reaccionó mejor al pico de marzo de 2026, y su ventaja se concentra en dos episodios. La mayor parte de la información predecible está en la persistencia de la volatilidad, que también capta un promedio móvil simple.

In [9]:
!jupyter nbconvert --to notebook --execute --inplace "./notebooks/07.comparison_model.ipynb"

[NbConvertApp] Converting notebook ./notebooks/07.comparison_model.ipynb to notebook
C:\Users\ASUS\Desktop\Maestria IA USFQ\AprendizajeSupervisado\Proyectofinal_ml_wti\.venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 702491 bytes to notebooks\07.comparison_model.ipynb


## 3. Resumen de resultados


### 3.1 Datos y diagnóstico de la serie
- **Datos:** 4.233 sesiones del futuro WTI (`CL=F`) entre el 4-ene-2010 y el 30-oct-2026. No hay faltantes; los tres días hábiles sin dato corresponden a cierres extraordinarios del mercado (huracán Sandy en 2012 y duelo por George H. W. Bush en 2018). Se eliminaron las sesiones del 20 y 21 de abril de 2020, con precios negativos, para poder calcular el retorno logarítmico y el estimador de Parkinson.
- **División temporal:** desarrollo 2010–2024 (3.769 días, 10 folds de *Time Series Cross-Validation*) y test final 2025–oct 2026 (461 días). El diagnóstico se hizo solo con enero 2010 – junio 2019 (2.385 días), el entrenamiento común a todos los folds.
- **Diagnóstico de los retornos:**

| Característica | Prueba | Resultado | Implicación |
|---|---|---|---|
| Estacionariedad | ADF, PP, KPSS | ADF = −34,40 y PP = −51,96 rechazan la raíz unitaria; KPSS no rechaza la estacionariedad (*p* = 0,10) | Se modelan los retornos |
| Estacionalidad | Kruskal-Wallis por mes | *p* = 0,504 | Sin términos estacionales |
| Dependencia en la media | PACF, Ljung-Box, AIC/BIC | Solo el rezago 1 es relevante (PACF = −0,065) | Media simple |
| Dependencia en la varianza | Ljung-Box sobre $r_t^2$, ARCH-LM | ARCH-LM = 296,4; *p* ≈ 0 | Modelo de la familia GARCH |
| Distribución | Curtosis, Jarque-Bera | Curtosis en exceso de 2,92; se rechaza la normalidad | Errores t de Student |



### 3.2 Modelo econométrico


- Entre tres especificaciones EGARCH(1,1,1)-t se eligió la de **media constante**: diagnósticos residuales más favorables y criterios de información prácticamente iguales al mejor (diferencia de 1,6 puntos de AIC).
- Cumple los supuestos sobre los residuos estandarizados: sin autocorrelación (Ljung-Box, *p* > 0,05), sin efectos ARCH remanentes (ARCH-LM, *p* = 0,106), ajuste aceptable a la t (KS, *p* = 0,302; ν ≈ 8,6) y sin sesgo de signo al 5 % (*p* = 0,097).
- Volatilidad muy persistente (β ≈ 0,99) y asimétrica (γ ≈ −0,07 < 0 en todos los folds): las caídas del precio elevan más la volatilidad que las subidas.



### 3.3 Variables y selección


- Se construyeron **67 variables** por rezagos (20 rezagos de $r^2$, $\lvert r\rvert$ y Parkinson, más volatilidades y promedios móviles de 5, 21 y 63 días, y el retorno con signo), todas con información hasta τ−1.
- La selección por información mutua eligió **k = 16**: con menos variables se obtiene el mismo error, por la redundancia entre ellas. Todas las elegidas son medidas de volatilidad basadas en Parkinson o en la desviación móvil de los retornos; ningún rezago de $r^2$, $\lvert r\rvert$ ni el retorno con signo.
- En el Random Forest, el **Parkinson del día anterior** (importancia 0,49) y el **promedio de Parkinson de 5 días** (0,25) concentran la mayor parte de la importancia.
- El t-SNE muestra un gradiente entre días de volatilidad baja y alta, con una zona intermedia mezclada.



### 3.4 Random Forest


- Optuna (50 pruebas, TPE y poda por mediana) eligió `n_estimators = 350`, `max_depth = 7`, `min_samples_leaf = 10` y `max_features ≈ 0,96`. El hiperparámetro más influyente fue `min_samples_leaf` (importancia 0,73).
- Las curvas de diagnóstico muestran subajuste con árboles poco profundos o con hojas grandes, y estabilidad del error de validación a partir de una profundidad de 6–7. Los valores elegidos están en la zona de menor error de validación.
- La curva de aprendizaje se estabiliza a partir de unos 1.000 días de entrenamiento (QLIKE de validación ≈ 2,85).



### 3.5 Comparación de modelos



| | AR-EGARCH | Random Forest | Ingenuo (media 21 días) |
|---|---|---|---|
| QLIKE entrenamiento | 2,374 | 2,297 | 2,379 |
| QLIKE validación (media de 10 folds) | 2,866 | 2,849 | 2,829 |
| QLIKE test 2025–2026 | 3,239 | **2,839** | 2,974 |
| MSE validación (media) | **989,2** | 1.324,4 | 1.128,9 |
| MSE test | 815,4 | **729,4** | 784,2 |
| Folds con menor QLIKE (EGARCH vs. RF) | 3 de 10 | 7 de 10 | — |


| Prueba | Comparación | Resultado |
|---|---|---|
| Wilcoxon (10 folds) | EGARCH vs. RF | QLIKE *p* = 0,492; MSE *p* = 0,084 |
| *t* pareada (10 folds) | EGARCH vs. RF | QLIKE *p* = 0,826; MSE *p* = 0,137 |
| Diebold-Mariano (validación, por repetición) | EGARCH vs. RF | QLIKE *p* = 0,724 y 0,853; MSE *p* = 0,092 y 0,089 |
| Diebold-Mariano (validación) | Cada modelo vs. ingenuo | *p* > 0,05 en todas las comparaciones |
| Diebold-Mariano (test) | EGARCH vs. RF | QLIKE *p* = 0,147; MSE *p* = 0,722 |

Ninguna diferencia es significativa al 5 %.

## 4. Conclusiones


1. **El diagnóstico justificó el modelo base.** Los retornos del WTI son estacionarios, sin estacionalidad mensual y con una dependencia débil en la media, pero con un agrupamiento de volatilidad muy marcado (ARCH-LM, *p* ≈ 0) y colas pesadas. Esto llevó a un modelo de la familia GARCH con errores t. El EGARCH de media constante cumplió los supuestos residuales y mostró una volatilidad muy persistente y asimétrica: las caídas del petróleo aumentan la volatilidad más que las subidas.

2. **Ningún modelo fue significativamente mejor.** El Random Forest obtuvo una QLIKE de validación ligeramente menor que el EGARCH (2,849 frente a 2,866) y fue mejor en 7 de 10 folds, pero la diferencia no es significativa ni con Wilcoxon (*p* = 0,492) ni con Diebold-Mariano (*p* ≥ 0,72 en QLIKE). Con el MSE, el EGARCH tuvo menor error medio, también sin significancia (*p* ≈ 0,08–0,09).

3. **La ventaja depende del episodio de volatilidad.** En validación, el Random Forest fue mejor en los periodos de volatilidad moderada (2021–2024) y el EGARCH en los dos folds con la crisis de 2020, donde el Random Forest subestimó los picos de marzo y abril. En el test ocurrió lo contrario: durante el pico de marzo de 2026, el Random Forest elevó su pronóstico mucho más rápido que el EGARCH. Ningún modelo responde sistemáticamente mejor a los eventos extremos; los dos resultan complementarios.

4. **La información útil está en la volatilidad reciente.** La selección de variables y la importancia en el Random Forest coinciden en que lo más informativo son el estimador de Parkinson de los días anteriores y sus promedios de corto plazo; los rezagos de los retornos al cuadrado no se seleccionaron. Además, un promedio de Parkinson de 21 días obtuvo la QLIKE de validación más baja de los tres (2,829), sin diferencias significativas. La mayor parte de lo que se puede pronosticar de la volatilidad diaria proviene de su persistencia, que también capta una referencia simple.

5. **El test final mantiene el orden descriptivo.** En 2025–2026, el Random Forest volvió a tener las menores pérdidas (QLIKE 2,839; MSE 729,4), sin diferencias significativas frente al EGARCH (*p* = 0,147). Su error de test fue casi igual al de validación, de modo que la validación cruzada anticipó bien su desempeño. El EGARCH empeoró en el test (de 2,866 a 3,239): su alta persistencia hace que la varianza pronosticada cambie poco ante un choque de un día, por lo que subestimó el pico de marzo de 2026, y la QLIKE penaliza fuertemente la subestimación. La ventaja del Random Forest en el test se concentra en dos episodios (junio de 2025 y marzo de 2026).

6. **No hay señales de sobreajuste severo.** Ambos modelos presentan una brecha similar entre entrenamiento y validación (0,49 en el EGARCH y 0,55 en el Random Forest), y la mayor parte de la variación del error se debe a las diferencias entre periodos, no a los hiperparámetros.

7. **Respuesta a la pregunta de investigación.** ¿Un Random Forest basado en rezagos de medidas de volatilidad pronostica la volatilidad diaria del WTI con mayor precisión que un EGARCH(1,1,1) con media constante y errores t, y es esa diferencia significativa? 

El Random Forest obtuvo pérdidas ligeramente menores en validación y en el test, pero ni la prueba de Wilcoxon ni la de Diebold-Mariano encontraron diferencias significativas. Por tanto, **no se rechaza $H_0$**: no hay evidencia suficiente para afirmar que un modelo tenga mayor precisión que el otro. Esto no demuestra que sean equivalentes; indica que la diferencia observada es pequeña frente a la variabilidad entre periodos. El EGARCH es más parsimonioso e interpretable; el Random Forest no requiere supuestos sobre la forma de la varianza y su desempeño fue más estable entre validación y test.

**Limitaciones.**

- La varianza real del mercado no se observa directamente, así que el análisis usa el estimador de Parkinson como una aproximación (proxy).
Parkinson calcula la variación intradía a partir del precio máximo y mínimo del día. Es “ruidoso” porque esa estimación puede diferir de la volatilidad real: depende de los precios observados y no recoge todos los movimientos del mercado.  Además, como usa el máximo y el mínimo dentro de la sesión, no incorpora el salto que puede ocurrir entre el cierre de un día y la apertura del siguiente.  Por lo tanto, el Parkinson sirve como medida aproximada de volatilidad intradía, pero puede omitir movimientos entre sesiones y contener error de medición.

- `CL=F` es una serie continua que encadena contratos de futuros, por lo que puede presentar saltos en los cambios de contrato, y los episodios extremos de 2020 están ligados al vencimiento de un contrato específico.
- La serie atraviesa cambios de régimen (2014–2016, 2020, 2022) que afectan de forma distinta a cada modelo.
- Los folds de las dos repeticiones se superponen en el tiempo, por lo que los 10 pares de errores no son completamente independientes y las pruebas tienen poca potencia.

- Los hiperparámetros y el número de variables se eligieron con los mismos 10 folds que se usan para comparar, por lo que la QLIKE de validación del Random Forest puede ser algo optimista; el test final, no usado en ninguna decisión, sirve como control.
- La ventaja del Random Forest en el test depende de pocos episodios de volatilidad extrema, lo que limita la potencia de las pruebas.

**Trabajo futuro.**
- Combinar los pronósticos de ambos modelos, por ejemplo ponderándolos según el régimen de volatilidad.
- Comparar con un modelo HAR, la referencia habitual cuando se usan medidas de volatilidad realizada.
- Incorporar variables exógenas, como inventarios de crudo, el índice del dólar o el VIX.
- Usar medidas de volatilidad realizada con datos intradía, menos ruidosas que el estimador de Parkinson.

## 5. Referencias
[1] R. F. Engle, "Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation," *Econometrica*, vol. 50, no. 4, pp. 987–1007, 1982.

[2] T. Bollerslev, "Generalized autoregressive conditional heteroskedasticity," *J. Econometrics*, vol. 31, no. 3, pp. 307–327, 1986.

[3] P. R. Hansen y A. Lunde, "A forecast comparison of volatility models: Does anything beat a GARCH(1,1)?," *J. Appl. Econometrics*, vol. 20, no. 7, pp. 873–889, 2005.

[4] L. Breiman, "Random forests," *Mach. Learn.*, vol. 45, no. 1, pp. 5–32, 2001.

[5] M. Parkinson, "The extreme value method for estimating the variance of the rate of return," *J. Business*, vol. 53, no. 1, pp. 61–65, 1980.

[6] A. J. Patton, "Volatility forecast comparison using imperfect volatility proxies," *J. Econometrics*, vol. 160, no. 1, pp. 246–256, 2011.

[7] F. X. Diebold y R. S. Mariano, "Comparing predictive accuracy," *J. Bus. Econ. Statist.*, vol. 13, no. 3, pp. 253–263, 1995.

_Agregar: referencias del software (scikit-learn, arch, statsmodels, Optuna, yfinance) y de cualquier otra fuente usada. Si se usó un asistente de IA para generar código, colocar los prompts como comentarios de Python en la celda correspondiente._